# EW-Mode Sentinel-1 Patch Extraction (Tuktoyaktuk)

Per Michel's suggestion: build a model conditioned on Sentinel-1 **EW**
(Extra Wide swath) data instead of IW, training in Tuktoyaktuk first,
with a follow-up notebook testing on Pond Inlet/Cambridge Bay. EW mode
turns out to have dramatically better temporal coverage of these Arctic
sites than IW -- the nearest EW scene to Tuktoyaktuk's survey date
(`2024-04-16`) is **0.7 days away**, versus IW/PC-RTC needing a
compromise of several days to weeks for the pcrtc track.

## Why this notebook looks different from `02`/`11`/`12`

Unlike PC-RTC (already radiometrically corrected and georeferenced),
Planetary Computer's EW data comes from the plain `sentinel-1-grd`
collection: **raw uint16 digital numbers**, no calibration applied, and
**no simple CRS/transform** -- geolocation is via embedded Ground
Control Points (GCPs), the same situation as the project's raw-SAFE
Track A/B data. This notebook reuses the calibration + GCP-warping
logic already built and proven in
`raw_data/03_sentinel1_preprocessing.ipynb` (`calibration_lut`,
`warp_grid`, `calibrate_and_warp`), adapted to read directly from
Planetary Computer's remote asset URLs instead of local SAFE folders --
the calibration math (`sigma0 = DN^2 / LUT^2`) and GCP-based
reprojection are unchanged.

## Resolution note

EW's native pixel spacing is **40m** (versus IW's 10m), so a 256m LiDAR
patch spans only ~6x6 real pixels here (`round(256/40)=6`), versus IW's
26x26. This is a genuine information-density limitation worth stating
plainly in the write-up -- the model's existing bilinear upsample to
256x256 (in the dataset adapter) will still run, but it's stretching
far less real spatial detail than the IW conditioning ever had.

## Setup

In [1]:
# Imports for calibration (adapted from raw_data/03), GCP-warping, and PC catalog access
import os
import io
import json
import datetime as dt
from datetime import timezone
import glob as glob_module
from pathlib import Path
from collections import Counter

import numpy as np
import requests
import rasterio
from rasterio.windows import Window, from_bounds
from rasterio.warp import transform_bounds, calculate_default_transform, reproject, Resampling
from scipy.interpolate import RectBivariateSpline
import xml.etree.ElementTree as ET

import pystac_client
import planetary_computer
from dotenv import load_dotenv

load_dotenv()
if os.environ.get('PC_SDK_SUBSCRIPTION_KEY'):
    planetary_computer.settings.set_subscription_key(os.environ['PC_SDK_SUBSCRIPTION_KEY'])

## Configuration

In [2]:
# Region-specific paths and EW-specific resolution/patch-size constants
REPO_DIR = Path('/cs/student/project_msc/2025/aibh/jiayiche')
INPUT_DIR = REPO_DIR / 'input_data'
REGION = 'tuk'
LIDAR_DIR = INPUT_DIR / 'lidar_patches_tuk_tessa'
SURVEY_DATE = dt.date(2024, 4, 16)
N_SCENES = 3  # matches CONTEXT_K used throughout this project

PATCH_SIZE = 256
EW_PIXEL_SPACING_M = 40.0
EW_S1_PATCH_SIZE = int(round(PATCH_SIZE / EW_PIXEL_SPACING_M))  # 6 px @ 40m

CALIBRATED_DIR = REPO_DIR / 'raw_data' / f'{REGION}_ew_calibrated'
OUT_S1_DIR = INPUT_DIR / f's1_patches_{REGION}_ew'
CALIBRATED_DIR.mkdir(parents=True, exist_ok=True)
OUT_S1_DIR.mkdir(parents=True, exist_ok=True)

with rasterio.open(next(LIDAR_DIR.glob('lidar_patch_*.tif'))) as ref:
    DST_CRS = ref.crs
print('LIDAR_DIR:', LIDAR_DIR)
print('DST_CRS (from LiDAR patches):', DST_CRS)
print('EW_S1_PATCH_SIZE:', EW_S1_PATCH_SIZE, f'px @ {EW_PIXEL_SPACING_M}m (target patch is {PATCH_SIZE}m)')
print('OUT_S1_DIR:', OUT_S1_DIR)

LIDAR_DIR: /cs/student/project_msc/2025/aibh/jiayiche/input_data/lidar_patches_tuk_tessa
DST_CRS (from LiDAR patches): EPSG:32608
EW_S1_PATCH_SIZE: 6 px @ 40.0m (target patch is 256m)
OUT_S1_DIR: /cs/student/project_msc/2025/aibh/jiayiche/input_data/s1_patches_tuk_ew


## 1. Select the nearest EW-mode scenes

Already confirmed via direct query: the nearest EW scene to
`2024-04-16` is 0.7 days away. This re-runs that same search and picks
the `N_SCENES` closest, HH+HV, chronologically ordered for `t0/t1/t2`
naming.

In [3]:
# Build the Tuktoyaktuk AOI from the LiDAR patches, then search sentinel-1-grd for the nearest EW-mode scenes
from concurrent.futures import ThreadPoolExecutor, as_completed
from shapely.geometry import box, shape
from shapely.ops import unary_union
from rasterio.warp import transform_geom

def aoi_from_lidar_patches(patches_dir, max_files=300, workers=8):
    paths = sorted(patches_dir.glob('lidar_patch_*.tif'))
    if len(paths) > max_files:
        stride = len(paths) / max_files
        paths = [paths[int(i * stride)] for i in range(max_files)]
    def read_bounds(path):
        with rasterio.open(path) as src:
            return src.crs, src.bounds
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(read_bounds, p) for p in paths]
        for future in as_completed(futures):
            results.append(future.result())
    crs = results[0][0]
    native = unary_union([box(*bounds) for _, bounds in results])
    geojson = transform_geom(crs, 'EPSG:4326', native.__geo_interface__)
    return shape(geojson).buffer(0)

aoi = aoi_from_lidar_patches(LIDAR_DIR)
aoi_ll = aoi.convex_hull
print('AOI bounds (lon, lat):', aoi_ll.bounds)

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
search = catalog.search(collections=['sentinel-1-grd'], intersects=aoi_ll.__geo_interface__)
all_items = list(search.items())
ew_items = [it for it in all_items if it.properties.get('sar:instrument_mode') == 'EW']
print(f'Total sentinel-1-grd scenes: {len(all_items)} | EW-mode: {len(ew_items)}')

target = dt.datetime.combine(SURVEY_DATE, dt.time(), tzinfo=timezone.utc)
items = sorted(ew_items, key=lambda it: abs((it.datetime - target).total_seconds()))[:N_SCENES]
items = sorted(items, key=lambda it: it.datetime)

print(f'\nSelected {len(items)} nearest EW scenes to {SURVEY_DATE.isoformat()}:')
for i, item in enumerate(items):
    days_off = (item.datetime - target).total_seconds() / 86400
    print(f'  t{i}: {item.id} | {item.datetime} | {days_off:+.1f} days | pols={item.properties.get("sar:polarizations")}')

AOI bounds (lon, lat): (-133.36095441893116, 69.69849720309567, -133.27492950651194, 69.85188928067103)
Total sentinel-1-grd scenes: 1551 | EW-mode: 1007

Selected 3 nearest EW scenes to 2024-04-16:
  t0: S1A_EW_GRDM_1SDH_20240413T152939_20240413T153037_053417_067AC7 | 2024-04-13 15:30:08.508089+00:00 | -2.4 days | pols=['HH', 'HV']
  t1: S1A_EW_GRDM_1SDH_20240416T155420_20240416T155457_053461_067C7E | 2024-04-16 15:54:39.309721+00:00 | +0.7 days | pols=['HH', 'HV']
  t2: S1A_EW_GRDM_1SDH_20240418T153755_20240418T153853_053490_067DA3 | 2024-04-18 15:38:24.305300+00:00 | +2.7 days | pols=['HH', 'HV']


## 2. Calibration + GCP-warping helpers

Adapted from `raw_data/03_sentinel1_preprocessing.ipynb`, reading from
remote Planetary Computer URLs instead of local SAFE files. The
calibration math and GCP-based reprojection are unchanged -- only the
data source (a signed HTTPS COG URL for the measurement band, an HTTP
GET for the calibration XML) differs.

**Destination grid is sized to the AOI, not the whole scene.**
`calculate_default_transform` on the source's full width/height (as
`raw_data/03` did, fine there since IW scenes are narrower) would size
the output to the *entire* ~400km EW swath here -- confirmed directly:
a first attempt computed a 12166x11679 output (~487km x 467km) for an
AOI that's only ~3km x 17km. `compute_dst_grid_from_aoi` instead builds
the destination transform/width/height directly from the known AOI
bounds (with a small pixel buffer), independent of the source scene's
size -- computed once and reused for every scene/band, since it no
longer depends on any per-scene raster property.

In [4]:
# Calibration LUT + GCP-warping, adapted from raw_data/03 -- destination grid now sized to the AOI, not the whole scene
def calibration_lut_from_url(xml_url, shape):
    response = requests.get(xml_url)
    response.raise_for_status()
    root = ET.fromstring(response.content)
    vectors = root.find('calibrationVectorList').findall('calibrationVector')
    lines, pixels, values = [], None, []
    for vector in vectors:
        lines.append(int(vector.find('line').text))
        current_pixels = np.array([int(x) for x in vector.find('pixel').text.split()], dtype=float)
        pixels = current_pixels if pixels is None else pixels
        values.append([float(x) for x in vector.find('sigmaNought').text.split()])
    interpolator = RectBivariateSpline(np.asarray(lines, float), pixels, np.asarray(values, float), kx=1, ky=1)
    return interpolator(np.arange(shape[0]), np.arange(shape[1])).astype(np.float32)


def compute_dst_grid_from_aoi(aoi_bounds_wgs84, dst_crs, resolution, buffer_px=10):
    left, bottom, right, top = transform_bounds('EPSG:4326', dst_crs, *aoi_bounds_wgs84)
    buffer = resolution * buffer_px
    left -= buffer; bottom -= buffer; right += buffer; top += buffer
    width = int(np.ceil((right - left) / resolution))
    height = int(np.ceil((top - bottom) / resolution))
    dst_transform = rasterio.transform.from_origin(left, top, resolution, resolution)
    return dst_transform, width, height


def calibrate_and_warp_remote(measurement_href, calibration_xml_url, dst_crs, output_path, dst_transform, dst_width, dst_height):
    with rasterio.open(measurement_href) as src:
        dn = src.read(1).astype(np.float32)
        gcps, gcp_crs = src.gcps
    lut = calibration_lut_from_url(calibration_xml_url, dn.shape)
    with np.errstate(divide='ignore', invalid='ignore'):
        sigma0 = np.where(lut > 0, (dn ** 2) / (lut ** 2), np.nan).astype(np.float32)
    profile = {'driver': 'GTiff', 'dtype': 'float32', 'count': 1, 'width': dst_width, 'height': dst_height,
               'crs': dst_crs, 'transform': dst_transform, 'nodata': np.nan, 'compress': 'deflate'}
    with rasterio.open(output_path, 'w', **profile) as dst:
        reproject(sigma0, rasterio.band(dst, 1), gcps=gcps, src_crs=gcp_crs,
                  dst_transform=dst_transform, dst_crs=dst_crs, resampling=Resampling.bilinear,
                  src_nodata=np.nan, dst_nodata=np.nan)
    return output_path


# Computed once, reused for every scene/band -- no longer depends on any per-scene source raster property
DST_TRANSFORM, DST_WIDTH, DST_HEIGHT = compute_dst_grid_from_aoi(aoi_ll.bounds, DST_CRS, EW_PIXEL_SPACING_M)
print(f'Destination grid: {DST_WIDTH} x {DST_HEIGHT} px @ {EW_PIXEL_SPACING_M}m '
      f'({DST_WIDTH * EW_PIXEL_SPACING_M / 1000:.2f}km x {DST_HEIGHT * EW_PIXEL_SPACING_M / 1000:.2f}km)')

Destination grid: 115 x 450 px @ 40.0m (4.60km x 18.00km)


### Sanity check on one scene before running all three

Same discipline used throughout this project -- verify the calibration
and warp actually produce sane output on a single scene before
committing to the full batch.

In [5]:
# Verify calibration + warp on t0's HH band alone before running the full batch
sample_item = items[0]
sample_out = CALIBRATED_DIR / '_sanity_check_hh.tif'
calibrate_and_warp_remote(sample_item.assets['hh'].href, sample_item.assets['schema-calibration-hh'].href,
                           DST_CRS, sample_out, DST_TRANSFORM, DST_WIDTH, DST_HEIGHT)
with rasterio.open(sample_out) as src:
    arr = src.read(1)
    valid = arr[np.isfinite(arr)]
    print('Shape:', arr.shape)
    print('Finite fraction:', np.isfinite(arr).mean())
    print('Valid-pixel min/max/mean (calibrated sigma0, linear):', valid.min(), valid.max(), valid.mean())
sample_out.unlink()

Shape: (450, 115)
Finite fraction: 1.0
Valid-pixel min/max/mean (calibrated sigma0, linear): 0.0022373477 0.061051887 0.013595442


## 3. Calibrate + warp all three scenes, stack HH+HV

Same `t{i}.tif` 2-band convention used throughout this project.

In [6]:
# Calibrate + warp HH and HV for each selected scene, stack into the standard t{i}.tif 2-band convention
merged_paths = []
merged_attrs = []

for i, item in enumerate(items):
    hh_out = CALIBRATED_DIR / f'_hh_{i}.tif'
    hv_out = CALIBRATED_DIR / f'_hv_{i}.tif'
    calibrate_and_warp_remote(item.assets['hh'].href, item.assets['schema-calibration-hh'].href,
                               DST_CRS, hh_out, DST_TRANSFORM, DST_WIDTH, DST_HEIGHT)
    calibrate_and_warp_remote(item.assets['hv'].href, item.assets['schema-calibration-hv'].href,
                               DST_CRS, hv_out, DST_TRANSFORM, DST_WIDTH, DST_HEIGHT)

    final = CALIBRATED_DIR / f't{i}.tif'
    with rasterio.open(hh_out) as a, rasterio.open(hv_out) as b:
        meta = a.meta.copy(); meta.update(count=2)
        with rasterio.open(final, 'w', **meta) as dst:
            dst.write(a.read(1), 1)
            dst.write(b.read(1), 2)
    hh_out.unlink()
    hv_out.unlink()
    merged_paths.append(str(final))

    props = item.properties
    merged_attrs.append({
        'acquisition_date': item.datetime.date().isoformat(),
        'orbit_direction': 'ASCENDING' if props.get('sat:orbit_state') == 'ascending' else 'DESCENDING',
        'relative_orbit_number': props.get('sat:relative_orbit'),
    })
    with rasterio.open(final) as f:
        arr = f.read()
        print(f'Wrote t{i}.tif: shape={arr.shape}, finite_frac={np.isfinite(arr).mean():.4f}')

attrs_json_path = CALIBRATED_DIR / 'attrs.json'
with open(attrs_json_path, 'w') as jf:
    json.dump(merged_attrs, jf, indent=2)
print('Wrote', attrs_json_path)

Wrote t0.tif: shape=(2, 450, 115), finite_frac=1.0000
Wrote t1.tif: shape=(2, 450, 115), finite_frac=1.0000
Wrote t2.tif: shape=(2, 450, 115), finite_frac=1.0000
Wrote /cs/student/project_msc/2025/aibh/jiayiche/raw_data/tuk_ew_calibrated/attrs.json


## 4. Match against Tuktoyaktuk's existing LiDAR patches

In [7]:
# Same matching logic used throughout this project -- only s1_patch_size differs (6px @ 40m instead of 26px @ 10m)
def build_s1_products_from_corrected(geotiff_paths, attrs_jsons=None):
    products = []
    for i, path in enumerate(geotiff_paths):
        src = rasterio.open(path)
        attrs = attrs_jsons[i] if attrs_jsons and i < len(attrs_jsons) else None
        products.append({"src": src, "crs": src.crs, "transform": src.transform,
                          "height": src.height, "width": src.width, "attrs": attrs})
    if not products:
        raise ValueError("No Sentinel-1 products loaded -- check geotiff_paths.")
    return products


def close_products(products):
    for p in products:
        p["src"].close()


def extract_lidar_matched_s1_patches(lidar_patches_dir, sentinel1_products, s1_patch_size,
                                      out_s1_dir, pattern="lidar_patch_*.tif", max_nan_frac=0.02):
    lidar_paths = sorted(glob_module.glob(os.path.join(str(lidar_patches_dir), pattern)))
    print(f"Found {len(lidar_paths)} existing LiDAR patches to match against "
          f"{len(sentinel1_products)} Sentinel-1 product(s).")

    n_written, n_skipped, n_skipped_nan = 0, 0, 0

    for idx, lp in enumerate(lidar_paths):
        if idx % 200 == 0:
            print(f"  ...processed {idx}/{len(lidar_paths)} "
                  f"(written: {n_written}, skipped: {n_skipped}, skipped-NaN: {n_skipped_nan})")

        patch_id = os.path.splitext(os.path.basename(lp))[0].split("_")[-1]

        with rasterio.open(lp) as lsrc:
            lidar_bounds = lsrc.bounds
            lidar_crs = lsrc.crs

        s1_patches, s1_transforms = [], []
        ok = True
        has_too_much_nan = False
        for prod in sentinel1_products:
            try:
                s1_bounds = transform_bounds(lidar_crs, prod["crs"], *lidar_bounds, densify_pts=21)
                window = from_bounds(*s1_bounds, transform=prod["transform"]).round_offsets().round_lengths()
                r0, c0 = int(window.row_off), int(window.col_off)
                hh, ww = int(window.height), int(window.width)

                if (hh, ww) != (s1_patch_size, s1_patch_size):
                    ok = False; break
                if r0 < 0 or c0 < 0:
                    ok = False; break
                if r0 + s1_patch_size > prod["height"] or c0 + s1_patch_size > prod["width"]:
                    ok = False; break

                read_window = Window(c0, r0, s1_patch_size, s1_patch_size)
                patch = prod["src"].read(window=read_window)
                if patch.shape[1:] != (s1_patch_size, s1_patch_size):
                    ok = False; break

                nan_frac = float(np.mean(np.isnan(patch)))
                if nan_frac > max_nan_frac:
                    has_too_much_nan = True
                    break

                s1_patches.append(patch)
                s1_transforms.append(rasterio.windows.transform(read_window, prod["transform"]))
            except Exception:
                ok = False
                break

        if has_too_much_nan:
            n_skipped_nan += 1
            continue

        if not ok or len(s1_patches) != len(sentinel1_products):
            n_skipped += 1
            continue

        patch_dir = os.path.join(str(out_s1_dir), f"s1_patch_{patch_id}")
        os.makedirs(patch_dir, exist_ok=True)

        attrs_list = []
        for ti, (prod, patch, tr) in enumerate(zip(sentinel1_products, s1_patches, s1_transforms)):
            meta = {
                "driver": "GTiff", "count": patch.shape[0],
                "height": s1_patch_size, "width": s1_patch_size,
                "dtype": "float32", "crs": prod["crs"], "transform": tr,
            }
            with rasterio.open(os.path.join(patch_dir, f"t{ti}.tif"), "w", **meta) as dst:
                dst.write(patch.astype(np.float32))
            attrs_list.append(prod.get("attrs"))

        with open(os.path.join(patch_dir, "attrs.json"), "w") as jf:
            json.dump(attrs_list, jf, indent=2)

        n_written += 1

    print(f"Done. Matched: {n_written}, skipped (out of bounds/wrong size): {n_skipped}, "
          f"skipped (too much NaN, >{max_nan_frac:.0%}): {n_skipped_nan}")
    return n_written, n_skipped

In [8]:
products_ew = build_s1_products_from_corrected(merged_paths, attrs_jsons=merged_attrs)
extract_lidar_matched_s1_patches(LIDAR_DIR, products_ew, EW_S1_PATCH_SIZE, OUT_S1_DIR)
close_products(products_ew)


Found 1676 existing LiDAR patches to match against 3 Sentinel-1 product(s).
  ...processed 0/1676 (written: 0, skipped: 0, skipped-NaN: 0)
  ...processed 200/1676 (written: 200, skipped: 0, skipped-NaN: 0)
  ...processed 400/1676 (written: 400, skipped: 0, skipped-NaN: 0)
  ...processed 600/1676 (written: 600, skipped: 0, skipped-NaN: 0)
  ...processed 800/1676 (written: 800, skipped: 0, skipped-NaN: 0)
  ...processed 1000/1676 (written: 1000, skipped: 0, skipped-NaN: 0)
  ...processed 1200/1676 (written: 1200, skipped: 0, skipped-NaN: 0)
  ...processed 1400/1676 (written: 1400, skipped: 0, skipped-NaN: 0)
  ...processed 1600/1676 (written: 1600, skipped: 0, skipped-NaN: 0)
Done. Matched: 1676, skipped (out of bounds/wrong size): 0, skipped (too much NaN, >2%): 0


## 5. Verify the output

In [9]:
# Same verification checks used throughout this project
sample_patches = sorted(glob_module.glob(os.path.join(str(OUT_S1_DIR), 's1_patch_*')))
lidar_total = len(glob_module.glob(os.path.join(str(LIDAR_DIR), 'lidar_patch_*.tif')))
print(f'Total patches written: {len(sample_patches)} (out of {lidar_total} Tuktoyaktuk LiDAR patches)')

if sample_patches:
    with rasterio.open(os.path.join(sample_patches[0], 't0.tif')) as src:
        print('Sample patch shape:', src.shape, f'(expect {EW_S1_PATCH_SIZE}x{EW_S1_PATCH_SIZE})')
    for f in sorted(glob_module.glob(os.path.join(sample_patches[0], 't*.tif'))):
        with rasterio.open(f) as src:
            arr = src.read()
            print(f'  {os.path.basename(f)}: finite_frac={np.isfinite(arr).mean():.4f}, '
                  f'nonzero_frac={(arr != 0).mean():.4f}, min/max={np.nanmin(arr):.5f}/{np.nanmax(arr):.5f}')

counts = Counter(len(glob_module.glob(os.path.join(p, 't*.tif'))) for p in sample_patches)
print('Distribution of timesteps per patch:', dict(sorted(counts.items())))

Total patches written: 1676 (out of 1676 Tuktoyaktuk LiDAR patches)
Sample patch shape: (6, 6) (expect 6x6)
  t0.tif: finite_frac=1.0000, nonzero_frac=1.0000, min/max=0.00059/0.02347
  t1.tif: finite_frac=1.0000, nonzero_frac=1.0000, min/max=0.00362/0.05195
  t2.tif: finite_frac=1.0000, nonzero_frac=1.0000, min/max=0.00114/0.03116
Distribution of timesteps per patch: {3: 1676}


In [10]:
import glob as glob_module
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds

sample_lidar_path = sorted(glob_module.glob(str(LIDAR_DIR / 'lidar_patch_*.tif')))[0]
print('Sample LiDAR patch:', sample_lidar_path)

with rasterio.open(sample_lidar_path) as lsrc:
    lidar_bounds = lsrc.bounds
    lidar_crs = lsrc.crs
print('LiDAR bounds:', lidar_bounds)
print('LiDAR CRS:', lidar_crs)
print('LiDAR patch width/height (m):', lidar_bounds.right - lidar_bounds.left, lidar_bounds.top - lidar_bounds.bottom)

for i, mp in enumerate(merged_paths):
    with rasterio.open(mp) as prod_src:
        print(f'\n--- t{i} ({mp}) ---')
        print('  Product CRS:', prod_src.crs)
        print('  Product transform:', prod_src.transform)
        print('  Product shape (h,w):', prod_src.shape)
        print('  Product bounds:', prod_src.bounds)

        s1_bounds = transform_bounds(lidar_crs, prod_src.crs, *lidar_bounds, densify_pts=21)
        print('  LiDAR bounds reprojected to product CRS:', s1_bounds)

        raw_window = from_bounds(*s1_bounds, transform=prod_src.transform)
        print('  Raw (unrounded) window:', raw_window)
        window = raw_window.round_offsets().round_lengths()
        print('  Rounded window:', window)
        print(f'  Window height x width: {window.height} x {window.width} (expect {EW_S1_PATCH_SIZE} x {EW_S1_PATCH_SIZE})')
        print(f'  Window row_off/col_off: {window.row_off}, {window.col_off} (must be >= 0)')
        print(f'  row_off+size={window.row_off + EW_S1_PATCH_SIZE} vs product height={prod_src.height}')
        print(f'  col_off+size={window.col_off + EW_S1_PATCH_SIZE} vs product width={prod_src.width}')


Sample LiDAR patch: /cs/student/project_msc/2025/aibh/jiayiche/input_data/lidar_patches_tuk_tessa/lidar_patch_12000.tif
LiDAR bounds: BoundingBox(left=563363.768032709, bottom=7742606.336082096, right=563619.768032709, top=7742862.336082096)
LiDAR CRS: EPSG:32608
LiDAR patch width/height (m): 256.0 256.0

--- t0 (/cs/student/project_msc/2025/aibh/jiayiche/raw_data/tuk_ew_calibrated/t0.tif) ---
  Product CRS: EPSG:32608
  Product transform: | 40.00, 0.00, 562601.68|
| 0.00,-40.00, 7750693.55|
| 0.00, 0.00, 1.00|
  Product shape (h,w): (450, 115)
  Product bounds: BoundingBox(left=562601.6791232366, bottom=7732693.554556994, right=567201.6791232366, top=7750693.554556994)
  LiDAR bounds reprojected to product CRS: (563363.768032709, 7742606.336082096, 563619.768032709, 7742862.336082096)
  Raw (unrounded) window: Window(col_off=np.float64(19.052222736811018), row_off=np.float64(195.78046187243308), width=np.float64(6.399999999999636), height=np.float64(6.399999999994179))
  Rounded windo

In [11]:
import rasterio
import numpy as np

with rasterio.open(merged_paths[0]) as src:
    window = rasterio.windows.Window(19, 195, 6, 6)  # from the trace above: col_off=19, row_off=195
    patch = src.read(window=window)
    print('Patch shape:', patch.shape)
    print('NaN count:', np.isnan(patch).sum(), '/', patch.size)
    print('NaN fraction:', np.isnan(patch).mean())
    print(patch)



Patch shape: (2, 6, 6)
NaN count: 0 / 72
NaN fraction: 0.0
[[[0.0192624  0.01547642 0.0130767  0.01310941 0.01331717 0.01185847]
  [0.02347197 0.01984572 0.01646468 0.01743335 0.01904163 0.01673493]
  [0.02141543 0.02138729 0.01922746 0.01900601 0.02113551 0.01999348]
  [0.01676235 0.0176675  0.01696185 0.01580919 0.01665374 0.01741971]
  [0.01700803 0.01704423 0.01556027 0.01316688 0.01208806 0.01302066]
  [0.01566125 0.01645687 0.01461374 0.01231581 0.01126857 0.01171256]]

 [[0.00094258 0.00071281 0.00064597 0.000661   0.00072768 0.00091829]
  [0.00098535 0.00068712 0.00059334 0.00080683 0.0010573  0.00127512]
  [0.00131325 0.0011772  0.00109621 0.00112554 0.00132142 0.00155311]
  [0.00149344 0.00148298 0.00129868 0.00112866 0.00125535 0.00146325]
  [0.00141934 0.00134556 0.00108321 0.00093404 0.00103828 0.00115028]
  [0.00116801 0.00095305 0.0007401  0.00073771 0.0008812  0.00099011]]]


## Next step (separate notebook)

If the checks above look healthy, the next step is a training notebook
mirroring `pcrtc/09`'s architecture and spatial-block split, but with:
- `S1_DIR` pointing at `s1_patches_tuk_ew`
- `cond_channels = 4 * CONTEXT_K` still holds (HH/HV repeated to 4
  channels per view, same as VV/VH was)
- The dataset adapter's `F.interpolate(..., size=(256,256))` step now
  upsamples from a genuinely coarser ~6x6 source instead of IW's 26x26
  -- worth watching whether this limits model performance independent
  of anything else, given how little real spatial detail survives at
  EW's native resolution.